# W9 · Day 2 — Rerankers, Query Rewriting, LlamaIndex

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Day 1 you built hybrid retrieval (BM25 + Dense + RRF). Day 2 adds the
second stage that makes the top-K precise: **cross-encoder reranking**.
Then a brief tour of query rewriting patterns (HyDE, multi-query, step-back)
— these you'll try on your capstone as Stretch lab, not today.

We close with a brief LlamaIndex intro so you know what the framework
abstracts. Framing is even-handed — you now understand what it hides;
pick frameworks when you want composition, stay from-scratch when you
want control.

**Same corpus as Day 1** — 15 Acme Analytics Platform docs. Imported from
`wk09_pipeline.py`.

**Cost per full run:** ~$0.01 (embeddings for retrieval + LLM calls for
HyDE/multi-query demos).

**Notebook flow:**
- Cell 1: Setup + **HARD assertion cross-encoder cached** (yesterday's homework)
- Cell 2: Two-stage retrieval concept — recall then precision
- Cell 3: Bi-encoder vs cross-encoder architecture
- Cell 4: Reranker in code
- Cell 5: Full stack (hybrid + rerank) on the 6 test queries
- Cell 6: Latency budget — time each stage, size the k_candidates trade-off
- Cell 7: Query rewriting — HyDE (runnable demo)
- Cell 8: Query rewriting — Multi-query (runnable demo)
- Cell 9: Query rewriting — Step-back (verbal + code sketch)
- Cell 10: LlamaIndex — 20-line equivalent of what we built
- Cell 11: Wrap + Core+Stretch hand-off

---

## Cell 1 — Setup + cross-encoder check

Yesterday's homework was to `pip install sentence-transformers` and
trigger the ~80MB cross-encoder download. Cell 1 hard-asserts the model
is cached. If it's missing, this cell fails loudly with install instructions.

In [ ]:
import os
import sys
import time
from pathlib import Path

# ── Cross-encoder check — HARD ASSERTION ──
try:
    from sentence_transformers import CrossEncoder
except ImportError as e:
    raise ImportError(
        "\n\n"
        "═══════════════════════════════════════════════════════════════\n"
        "SENTENCE-TRANSFORMERS NOT INSTALLED\n"
        "═══════════════════════════════════════════════════════════════\n"
        "This notebook requires sentence-transformers (yesterday's homework).\n\n"
        "Run:\n"
        "   pip install sentence-transformers\n\n"
        "Then reload the kernel and re-run this cell.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from e

# Try loading — this will use the cache if the download already happened
print("Loading cross-encoder (from cache if pre-downloaded)...")
try:
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    # Verify it works with a quick scoring call
    _ = reranker.predict([('test query', 'test document')])
    print("✓ Cross-encoder loaded and functional.")
except Exception as e:
    raise RuntimeError(
        "\n\nCross-encoder model failed to load. Likely causes:\n"
        "  1. Model wasn't downloaded yesterday (do the Day 1 homework)\n"
        "  2. HuggingFace access blocked by your network — talk to your instructor\n"
        f"\nOriginal error: {e}"
    ) from e

# ── Standard env + corpus setup ──
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY"

sys.path.insert(0, str(Path.cwd()))
from wk09_pipeline import (
    load_acme_corpus, load_test_queries,
    simple_tokenize, build_bm25_index, bm25_search,
    rrf_fuse,
)

corpus  = load_acme_corpus()
queries = load_test_queries()
print(f"✓ Loaded {len(corpus)} docs and {len(queries)} test queries.")

---

## Cell 2 — Two-stage retrieval: recall then precision

Yesterday's hybrid retrieval gave you a **top-K with good recall** — the
right answer is somewhere in the top-10, most of the time. But is it at
top-1 or top-5? A user only sees the top-3.

**Two-stage retrieval:**

```
STAGE 1 — cheap, wide retrieval:
  hybrid_retrieve(query, k=10)
  ↓ 10 candidates, mostly relevant, order is rough

STAGE 2 — expensive, precise reranking:
  cross_encoder.score(query, candidate) for each of the 10
  ↓ reorder by rerank_score
  ↓ take top 3
```

**Why not just retrieve top-3 with hybrid?** Because retrieval is a rough
cut. A good top-3 requires SEEING more candidates and picking wisely.
Top-10 gives the reranker enough to work with.

**Why not run the cross-encoder on all 15 docs (or 1500)?** Because
cross-encoders are slow — one forward pass per (query, doc) pair. See
Cell 6 for the timing budget.

---

## Cell 3 — Bi-encoder vs Cross-encoder

Both are transformer models. The **architectural difference** is where the
text and the query meet:

**Bi-encoder (dense retrieval — text-embedding-3-small, sentence-BERT):**
```
query  → [Encoder A] → vec_q  ─┐
                                 ├→  cosine(vec_q, vec_d) → score
doc    → [Encoder B] → vec_d  ─┘
```
Query and doc are encoded **separately**. Doc vectors can be pre-computed
and cached. Fast at query time (one embed + N cosine ops). Good for retrieval.

**Cross-encoder (rerankers — ms-marco-MiniLM):**
```
(query, doc) → [Cross-Encoder] → score
```
Query and doc are fed through **together**. The model can attend across
query and doc tokens simultaneously — much richer signal. But you must
run one full forward pass per (query, doc) pair. Slow. Can't be cached.

**Trade-off summary:**

| Property | Bi-encoder | Cross-encoder |
|---|---|---|
| Query-time cost | O(1) embed + O(N) cosine | O(N) forward passes |
| Precision | Good | Better (usually significantly) |
| Doc cache | Yes (embeddings) | No |
| Corpus size viable | Millions | Hundreds |

**Two-stage retrieval exploits both:** bi-encoder for the wide first cut
(fast on millions), cross-encoder for the narrow second cut (precise on
the top 10).

---

## Cell 4 — Reranker in code

The `sentence-transformers` library's `CrossEncoder` is 3 lines. We already
loaded it in Cell 1 to verify the download. Wrap the scoring into a
`rerank()` function.

In [ ]:
def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """Score each (query, candidate.text) pair; return top_k by rerank_score.
    
    candidates: list of dicts with 'doc' key containing full doc (must have 'text').
    Adds 'rerank_score' to each returned dict.
    """
    pairs = [(query, hit["doc"]["text"]) for hit in candidates]
    scores = reranker.predict(pairs)
    scored = [{**hit, "rerank_score": float(s)} for hit, s in zip(candidates, scores)]
    scored.sort(key=lambda h: h["rerank_score"], reverse=True)
    return scored[:top_k]

# Sanity check — score a single (query, doc) pair from the corpus
query = "what's error code AC-1042 about"
candidate_correct = {"id": "err_ac1042", "doc": next(d for d in corpus if d["id"] == "err_ac1042")}
candidate_wrong   = {"id": "err_ac4408", "doc": next(d for d in corpus if d["id"] == "err_ac4408")}

scored = rerank(query, [candidate_correct, candidate_wrong], top_k=2)
print(f"Query: {query!r}\n")
print(f"Reranker scores (higher = more relevant):")
for hit in scored:
    print(f"  {hit['id']:15s}  score = {hit['rerank_score']:+.3f}")
print()
print("The cross-encoder correctly ranks the ABOUT-AC-1042 doc over the")
print("MENTIONS-AC-1042 doc — the semantic distinction BM25 alone missed.")

**Interpretation of cross-encoder scores:**
- MS MARCO cross-encoders output logits, not probabilities. Range is roughly
  -12 (definitely irrelevant) to +12 (definitely relevant).
- You compare scores WITHIN a set of candidates for the same query, not
  across queries.
- **Never mix cross-encoder scores with retriever scores in a formula** —
  they're incomparable magnitudes.

---

## Cell 5 — Full stack: hybrid + rerank on all 6 queries

Now the payoff. Same 6 queries as Day 1, run through the full stack.
Compare vs Day 1's hybrid-only numbers.

**Pipeline:**
```
query
  → hybrid_retrieve(k_per_retriever=10, k_final=10)   ← wide first cut
  → rerank(top_k=3)                                     ← precise second cut
  → top-3
```

In [ ]:
# Rebuild BM25 + Qdrant setup (Day 1's variables aren't in scope in Day 2)
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

openai_client = OpenAI()
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = "wk09_day2_acme"

try: qdrant.delete_collection(COLLECTION)
except Exception: pass

qdrant.create_collection(collection_name=COLLECTION,
                        vectors_config=VectorParams(size=1536, distance=Distance.COSINE))

# Embed + upsert
texts = [d["text"] for d in corpus]
vectors = [item.embedding for item in openai_client.embeddings.create(
    model="text-embedding-3-small", input=texts).data]
qdrant.upsert(collection_name=COLLECTION, points=[
    PointStruct(id=i, vector=v, payload=d) for i, (d, v) in enumerate(zip(corpus, vectors))
])

# Build BM25
bm25 = build_bm25_index(corpus)

def dense_search(query: str, k: int = 10) -> list[dict]:
    q_vec = openai_client.embeddings.create(
        model="text-embedding-3-small", input=[query]).data[0].embedding
    hits = qdrant.query_points(collection_name=COLLECTION, query=q_vec, limit=k).points
    return [{"id": h.payload["id"], "title": h.payload["title"], "score": h.score,
             "doc": h.payload} for h in hits]

def hybrid_retrieve(query: str, k_per_retriever: int = 10, k_final: int = 10) -> list[dict]:
    bm25_hits  = bm25_search(bm25, corpus, query, k=k_per_retriever)
    dense_hits = dense_search(query, k=k_per_retriever)
    return rrf_fuse([bm25_hits, dense_hits], k=60, top_n=k_final)

print("✓ Full stack ready: hybrid_retrieve → rerank")

In [ ]:
print(f"══ Full stack (hybrid → rerank) on all 6 queries (top-1) ══\n")
print(f"  {'Query':<50s}  {'Hybrid top-1':<18s}  {'Reranked top-1':<18s}  Verdict")
print(f"  {'-----':<50s}  {'------------':<18s}  {'--------------':<18s}  -------")

hybrid_correct   = 0
reranked_correct = 0

for tq in queries:
    hybrid_top10 = hybrid_retrieve(tq["q"], k_per_retriever=10, k_final=10)
    reranked_top3 = rerank(tq["q"], hybrid_top10, top_k=3)
    
    hybrid_top1 = hybrid_top10[0]["id"]
    rerank_top1 = reranked_top3[0]["id"]
    
    hybrid_ok   = hybrid_top1 == tq["expected_id"]
    rerank_ok   = rerank_top1 == tq["expected_id"]
    if hybrid_ok:   hybrid_correct += 1
    if rerank_ok:   reranked_correct += 1
    
    change = ""
    if not hybrid_ok and rerank_ok:  change = "  ← rerank FIXED"
    elif hybrid_ok and not rerank_ok: change = "  ← rerank BROKE (rare)"
    
    verdict = f"{'✓' if rerank_ok else '✗'} (expected {tq['expected_id']})"
    print(f"  {tq['q'][:50]:<50s}  {hybrid_top1:<18s}  {rerank_top1:<18s}  {verdict}{change}")

print()
print(f"══ Progression across all 3 stages ══")
print(f"  Hybrid retrieval (top-1):    {hybrid_correct}/{len(queries)}")
print(f"  + Cross-encoder rerank:      {reranked_correct}/{len(queries)}")

**The moment where the DANGER ZONE work pays off.** On queries where hybrid
had the right answer at rank 3-5, the reranker pulls it to rank 1. On
queries where hybrid missed entirely (both retrievers wrong), the reranker
can't help — those are candidates for query rewriting.

**Discussion moment:**
- Which queries did reranker fix vs hybrid?
- Look at the cross-reference case (`err_ac1042`) — did the reranker
  correctly promote the ABOUT-AC-1042 doc over MENTIONS-AC-1042?
- For any queries still wrong after rerank — what would you try? Rewriting
  the query? Adding a metadata filter? Better chunking upstream?

---

## Cell 6 — Latency budget: where the time goes

The precision lift is nice, but reranking has a cost. Time each stage so
you can size the trade-off.

Warning: local cross-encoder timings on Vocareum's shared CPU can vary a
lot. The RELATIVE proportions (retrieval fast, rerank slow) matter; the
absolute numbers less so.

In [ ]:
# Time the full pipeline over 3 runs to get a stable-ish read
test_query = "how do I export a dashboard to PDF"
N_RUNS = 3

stage_times = {"embed_query": [], "bm25": [], "dense": [], "rrf": [], "rerank": []}

for _ in range(N_RUNS):
    # Stage: embed the query
    t0 = time.time()
    q_vec = openai_client.embeddings.create(
        model="text-embedding-3-small", input=[test_query]).data[0].embedding
    stage_times["embed_query"].append(time.time() - t0)
    
    # Stage: BM25 search
    t0 = time.time()
    bm25_hits = bm25_search(bm25, corpus, test_query, k=10)
    stage_times["bm25"].append(time.time() - t0)
    
    # Stage: Qdrant dense search
    t0 = time.time()
    dense_hits = qdrant.query_points(collection_name=COLLECTION, query=q_vec, limit=10).points
    dense_hits = [{"id": h.payload["id"], "score": h.score, "doc": h.payload} for h in dense_hits]
    stage_times["dense"].append(time.time() - t0)
    
    # Stage: RRF fusion
    t0 = time.time()
    fused = rrf_fuse([bm25_hits, dense_hits], k=60, top_n=10)
    stage_times["rrf"].append(time.time() - t0)
    
    # Stage: cross-encoder rerank
    t0 = time.time()
    _ = rerank(test_query, fused, top_k=3)
    stage_times["rerank"].append(time.time() - t0)

print(f"══ Latency per stage (median of {N_RUNS} runs) ══\n")
print(f"  {'Stage':<15s}  {'Time (ms)':>10s}  Notes")
print(f"  {'-----':<15s}  {'---------':>10s}  -----")

def median(lst): return sorted(lst)[len(lst)//2]

notes = {
    "embed_query": "1 API call to OpenAI (network-bound)",
    "bm25":        "in-memory over 15 docs — trivial",
    "dense":       "1 API call to Qdrant (network-bound)",
    "rrf":         "pure Python arithmetic — free",
    "rerank":      "10 forward passes through CrossEncoder — CPU-bound",
}

total = 0
for stage, times in stage_times.items():
    ms = median(times) * 1000
    total += ms
    print(f"  {stage:<15s}  {ms:>8.1f}    {notes[stage]}")

print(f"  {'-'*15}  {'-'*9}")
print(f"  {'TOTAL':<15s}  {total:>8.1f} ms")

print()
print("Key observation: reranker is usually the slowest stage. Its cost scales")
print("LINEARLY with k_candidates. Doubling from 10 to 20 candidates ~doubles")
print("rerank time. That's why we pick k_candidates=10 (not 100) as default.")

**Latency trade-off in a table:**

| k_candidates | Rerank time (approx) | Recall (typical) |
|---|---|---|
| 3 | ~50 ms | 60% — too aggressive |
| 10 | ~150 ms | 90% — programme default |
| 20 | ~300 ms | 95% — worth it for high-stakes |
| 100 | ~1500 ms | 99% — rarely worth it |

**Programme default: k_candidates=10, top_k=3.** Change only if your KPIs
clearly say you should.

**Cleanup demo collection** (before moving on to query rewriting cells):

In [ ]:
# Keep the collection for the rest of the notebook (query rewriting cells still use it)
# We'll clean up at the very end.

---

## Cell 7 — Query rewriting #1: HyDE (Hypothetical Document Embeddings)

**The problem HyDE solves:** short or vague queries don't embed well.
'timeout' embeds as one concept; the doc talking about timeouts embeds
as ten concepts. Cosine similarity between them is dragged down by all
the surrounding meaning in the doc.

**HyDE trick:** ask the LLM to draft a hypothetical answer to the query,
then embed **that** and retrieve against it. The hypothetical answer
shares 'answer-shape' with the real doc, so their embeddings sit closer
in vector space.

**When HyDE wins:** short queries, one-liner questions, terminology gaps
between user language and doc language.

**When HyDE hurts:** you pay one extra LLM call per query (cost + latency).
For queries that already work fine, it adds nothing.

In [ ]:
def hyde_retrieve(query: str, k: int = 3) -> tuple[str, list[dict]]:
    """HyDE pattern: draft a fake answer, embed it, retrieve using it.
    Returns (hypothetical_answer, retrieved_docs)."""
    # 1. Ask the LLM for a hypothetical answer
    hyde_prompt = (
        f"Write a plausible one-paragraph answer to this question about a data "
        f"analytics platform. Don't hedge; write as if you know. If the question "
        f"asks about a specific product, feature, or error code, invent plausible "
        f"details.\n\nQuestion: {query}\n\nHypothetical answer:"
    )
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.3,
        messages=[{"role": "user", "content": hyde_prompt}],
    )
    hypothetical = resp.choices[0].message.content
    
    # 2. Embed the hypothetical and retrieve using it
    hyde_vec = openai_client.embeddings.create(
        model="text-embedding-3-small", input=[hypothetical]).data[0].embedding
    
    hits = qdrant.query_points(collection_name=COLLECTION, query=hyde_vec, limit=k).points
    docs = [{"id": h.payload["id"], "title": h.payload["title"], "score": h.score} for h in hits]
    return hypothetical, docs

# Demo on a query where dense alone struggled
test_q = "how do I get notified when something breaks"
print(f"Query: {test_q!r}\n")

# Baseline: plain dense retrieval
baseline = dense_search(test_q, k=3)
print(f"Baseline dense (top-3): {[h['id'] for h in baseline]}\n")

# HyDE
hypothetical, hyde_hits = hyde_retrieve(test_q, k=3)
print(f"LLM-drafted hypothetical answer:\n  {hypothetical}\n")
print(f"HyDE-retrieved (top-3): {[h['id'] for h in hyde_hits]}")

**Discussion:**
- Did HyDE return the same top-3 as baseline dense? Different order?
  Different docs entirely?
- **On this small corpus + short query**, HyDE often doesn't help much —
  the corpus is small enough that dense already finds the right doc.
- On larger corpora with very short queries (search-box style), HyDE
  becomes valuable.
- **Cost:** one extra LLM call per query. At $0.15 per 1M input tokens for
  gpt-4o-mini, that's ~$0.0001 per query. Cheap in isolation; adds up at
  scale.

---

## Cell 8 — Query rewriting #2: Multi-query

**The problem multi-query solves:** one query, one embedding, one retrieval.
If the user's phrasing is unlucky, you miss docs the model would have
retrieved for a differently-phrased version.

**Multi-query trick:** ask the LLM to rewrite the query 3 different ways.
Retrieve for each rewrite. Union the results (or run RRF over the ranked
lists).

**When multi-query wins:** ambiguous queries, queries where the corpus
uses different terminology than the user.

In [ ]:
def multi_query_retrieve(query: str, n_rewrites: int = 3, k: int = 5) -> tuple[list[str], list[dict]]:
    """Multi-query pattern: LLM rewrites → retrieve for each → RRF-fuse the ranked lists."""
    # 1. Get N rewrites from the LLM
    rewrite_prompt = (
        f"Rewrite this question in {n_rewrites} different ways that would return the same "
        f"answer. Vary terminology, sentence structure, and formality. Return each "
        f"rewrite on its own line — nothing else, no numbering.\n\nQuestion: {query}"
    )
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.4,
        messages=[{"role": "user", "content": rewrite_prompt}],
    )
    rewrites = [line.strip() for line in resp.choices[0].message.content.split("\n") if line.strip()]
    rewrites = rewrites[:n_rewrites]  # trim in case LLM over-produced
    
    # 2. Retrieve for each rewrite (also include the original query)
    all_queries = [query] + rewrites
    ranked_lists = [dense_search(q, k=k) for q in all_queries]
    
    # 3. RRF-fuse the ranked lists
    fused = rrf_fuse(ranked_lists, k=60, top_n=3)
    return rewrites, fused

# Demo on a query that could be phrased many ways
test_q = "my query is running slowly"
print(f"Query: {test_q!r}\n")

baseline = dense_search(test_q, k=3)
print(f"Baseline dense (top-3): {[h['id'] for h in baseline]}\n")

rewrites, mq_hits = multi_query_retrieve(test_q, n_rewrites=3, k=5)
print(f"LLM rewrites:")
for i, r in enumerate(rewrites, 1):
    print(f"  {i}. {r}")
print(f"\nMulti-query fused (top-3): {[h['id'] for h in mq_hits]}")

**Discussion:**
- What did the LLM rewrite the query to? Are the rewrites genuinely
  different or repetitive?
- Did multi-query surface different docs than baseline?
- **Cost warning:** multi-query is `n_rewrites + 1` retrievals per user query.
  4× retrievals + 1 extra LLM call. Careful at high query volumes.

---

## Cell 9 — Query rewriting #3: Step-back (verbal + code sketch)

**The problem step-back solves:** the user asks a specific question but
your corpus doesn't have a specific answer — only a general answer that
would help.

**Step-back trick:** ask the LLM to generate a MORE GENERAL version of
the question first, retrieve for the general version, then use both the
general and specific answers to compose the response.

**Example:**
- User: 'is AC-1042 fixed in v2.3.7?'
- Step-back: 'what is AC-1042 and how is it usually resolved?'
- Retrieval on the step-back version finds the AC-1042 doc even if v2.3.7
  isn't mentioned anywhere.

**Code sketch (not run — Stretch lab exercise):**

```python
def step_back_retrieve(query: str, k: int = 3) -> list[dict]:
    prompt = (
        f"Given a specific question, generate a more general version that would "
        f"return the background information needed to answer the specific question. "
        f"Return only the general question, no explanation.\n\n"
        f"Specific: {query}\nGeneral:"
    )
    # ... call LLM, get general_query
    # ... retrieve for both original AND general_query
    # ... union or RRF the results
```

**When step-back wins:** version-specific queries against version-agnostic
docs, jargon-heavy queries against plain-language docs.

**Programme framing:** try HyDE, multi-query, or step-back in the Stretch
lab. Pick ONE based on YOUR golden-set failure pattern — don't try all three.

---

## Cell 10 — LlamaIndex — 20-line equivalent

**We've built each retrieval component from scratch:** embedding, BM25,
RRF, cross-encoder. LlamaIndex is a framework that gives you the same
components pre-wired as composable objects.

**Framing (this is the important bit):**
- If you now understand what LlamaIndex abstracts, you understand which of
  its magic is real magic (well-tested composition) vs which is just glue
  around things you already built.
- **When to reach for LlamaIndex:** you want to try 5 retrieval variants
  quickly; you want production observability and evaluations built in; you
  want to plug in new components (rerankers, embeddings) without rewriring.
- **When to stay from-scratch:** you want to control every prompt exactly;
  you want to see the wire format between components; you don't want a
  dependency you can't easily upgrade or debug through.
- **Both are valid choices in industry.** Frameworks aren't for beginners;
  from-scratch isn't for masochists. It's a taste-and-scale decision.

**Below is what our hybrid+rerank pipeline looks like in LlamaIndex** (sketch
only, not run — LlamaIndex isn't in our dependency set):

In [ ]:
llamaindex_sketch = '''
# ── LlamaIndex equivalent of our hybrid+rerank pipeline (~20 lines) ──
# NOT RUN — sketch only. Would require: pip install llama-index

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.qdrant import QdrantVectorStore

# 1. Set up dense retriever (Qdrant)
vector_store = QdrantVectorStore(client=qdrant_client, collection_name="capstone_chunks_v2")
index = VectorStoreIndex.from_vector_store(vector_store)
dense_retriever = index.as_retriever(similarity_top_k=10)

# 2. Set up BM25 retriever
bm25_retriever = BM25Retriever.from_defaults(docstore=index.docstore, similarity_top_k=10)

# 3. Fuse them via LlamaIndex\'s built-in RRF
hybrid_retriever = QueryFusionRetriever(
    [dense_retriever, bm25_retriever],
    similarity_top_k=10,
    mode="reciprocal_rerank",  # RRF
)

# 4. Cross-encoder reranker as a postprocessor
reranker = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=3)

# 5. Use it
candidates = hybrid_retriever.retrieve("how do I export a dashboard")
final = reranker.postprocess_nodes(candidates, query_str="how do I export a dashboard")
for node in final:
    print(node.text[:100], node.score)
'''

print(llamaindex_sketch)
print("\nWhat it does: exactly what we built in Day 1 + Day 2 Cells 1-5.")
print("What it hides: RRF math, tokenization, cross-encoder wrapping.")
print("What you gain: composition. Swap dense_retriever, bm25_retriever, or reranker")
print("               freely without touching the fusion or postprocessing.")
print("What you lose: visibility into the wire format between stages.")

**We're NOT switching to LlamaIndex this week.** The programme stays
from-scratch through W12 (grounded Q&A with guardrails) at least. But
you should recognise the pattern when you meet it — most 'RAG framework'
articles you'll read online use LlamaIndex or LangChain as a starting
point.

---

## Cell 11 — Cleanup + Wrap

Delete the demo collection so your Qdrant cluster stays tidy.

In [ ]:
try:
    qdrant.delete_collection(COLLECTION)
    print(f"✓ Cleaned up demo collection: {COLLECTION}")
except Exception as e:
    print(f"Skip cleanup: {e}")

---

## Wrap: what you built across Day 1 + Day 2

**Day 1 — Hybrid search:**
1. Named 4 naive-RAG failure modes; mapped each to an upgrade
2. BM25 (TF + IDF + length norm) via `rank-bm25`
3. RRF (Reciprocal Rank Fusion) — the industry-standard way to combine ranked lists
4. `hybrid_retrieve()` = BM25 + Dense + RRF in ~10 lines

**Day 2 — Rerank + query rewriting + LlamaIndex:**
5. Two-stage retrieval: recall (wide) then precision (narrow)
6. Bi-encoder vs cross-encoder architectural distinction
7. `rerank()` with `cross-encoder/ms-marco-MiniLM-L-6-v2`
8. Full stack timings — reranker is the slow stage; k_candidates=10 is the default
9. Query rewriting patterns — HyDE (runnable), multi-query (runnable), step-back (sketch)
10. LlamaIndex framing — 20-line equivalent; even-handed choice

---

## Track B — Core + Stretch (first week of this tagging)

Open `AI-RAG_W9_Application_Growth_Guide.md`.

**🟢 CORE (mandatory · ~2.25 hours)** — everyone completes:
- Step 1: Add `src/rag/retrieval.py` with hybrid_retrieve + rerank; wire into `qdrant_rag.py`
- Step 2: Run full golden set through hybrid+rerank; measure precision@3, @5
- Step 3: Update `docs/kpi/wk9-snapshot.md` + ADR with new retrieval stack decisions

**🟡 STRETCH (optional · ~45 min)** — only if Core is solid:
- Step 4: Pick ONE query rewriting technique (HyDE / multi-query / step-back).
  Implement in `src/rag/query_rewrite.py`. Try on golden set. Keep-or-drop verdict.

**Expected KPI delta:** precision@3 up 5-20 percentage points vs W8 pure-dense.
If your delta is much smaller, tell your instructor — likely a chunking or
tokenisation issue upstream. If your delta is much larger, congratulate
yourself — your corpus really needed hybrid.

---

**Heads-up for W10:** back to normal pace. W10 is RAG Optimization + KB
Lifecycle — caching, evaluation harness, KB updates. The hard concept
peaks are behind you until W13-14 (agents). Consolidate over the weekend.